# Task 3 — Neural Machine Translation

In [ ]:
from itertools import zip_longest
from pathlib import Path
from tqdm.auto import tqdm
import re
from collections import Counter
import random
import math

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch_directml
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec


PROJECT_ROOT = Path.cwd().parent
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "preprocessed"

PREPROCESSED_EN_PATH = PREPROCESSED_DIR / "europarl_10pct_preprocessed.en"
PREPROCESSED_ES_PATH = PREPROCESSED_DIR / "europarl_10pct_preprocessed.es"

device = torch_directml.device()

print(f"Project root: {PROJECT_ROOT}")
print(f"English input: {PREPROCESSED_EN_PATH}")
print(f"Spanish input: {PREPROCESSED_ES_PATH}")

In [ ]:
WORD_PATTERN = re.compile(
    r"\w+(?:["’]\w+)*|[^\w\s]",
    flags=re.UNICODE,
)


def load_word_tokenized_pairs(
    english_path,
    spanish_path,
    max_pairs=None,
):
    """Load aligned preprocessed files and apply word tokenization."""
    tokenized_pairs = []

    with Path(english_path).open("r", encoding="utf-8") as en_file, \
         Path(spanish_path).open("r", encoding="utf-8") as es_file:

        for line_number, (en_line, es_line) in enumerate(
            zip_longest(en_file, es_file, fillvalue=None),
            start=1,
        ):
            if en_line is None or es_line is None:
                raise ValueError(
                    "The preprocessed files contain different numbers "
                    f"of lines near line {line_number}."
                )

            en_tokens = WORD_PATTERN.findall(
                en_line.rstrip("\r\n")
            )
            es_tokens = WORD_PATTERN.findall(
                es_line.rstrip("\r\n")
            )
            tokenized_pairs.append((en_tokens, es_tokens))

            if (
                max_pairs is not None
                and len(tokenized_pairs) >= max_pairs
            ):
                break

    return tokenized_pairs


def load_character_tokenized_pairs(
    english_path,
    spanish_path,
    max_pairs=None,
):
    """Load aligned preprocessed files and apply character tokenization."""
    tokenized_pairs = []

    with Path(english_path).open("r", encoding="utf-8") as en_file, \
         Path(spanish_path).open("r", encoding="utf-8") as es_file:

        for line_number, (en_line, es_line) in enumerate(
            zip_longest(en_file, es_file, fillvalue=None),
            start=1,
        ):
            if en_line is None or es_line is None:
                raise ValueError(
                    "The preprocessed files contain different numbers "
                    f"of lines near line {line_number}."
                )

            en_characters = list(en_line.rstrip("\r\n"))
            es_characters = list(es_line.rstrip("\r\n"))
            tokenized_pairs.append(
                (en_characters, es_characters)
            )

            if (
                max_pairs is not None
                and len(tokenized_pairs) >= max_pairs
            ):
                break

    return tokenized_pairs

In [ ]:
word_pairs = load_word_tokenized_pairs(
    PREPROCESSED_EN_PATH,
    PREPROCESSED_ES_PATH
)

character_pairs = load_character_tokenized_pairs(
    PREPROCESSED_EN_PATH,
    PREPROCESSED_ES_PATH
)

In [ ]:
train_val_pairs, test_pairs = train_test_split(
    word_pairs, test_size=0.20, random_state=42
)

train_pairs, val_pairs = train_test_split(
    train_val_pairs, test_size=0.15, random_state=42
)

number_of_word_pairs = len(word_pairs)
print("Number of Word Pairs Distribution:")
print(f"Train: {len(train_pairs)} ({len(train_pairs)/number_of_word_pairs*100:.2f}%) | Val: {len(val_pairs)} ({len(val_pairs)/number_of_word_pairs*100:.2f}%) | Test: {len(test_pairs)} ({len(test_pairs)/number_of_word_pairs*100:.2f}%)")

In [ ]:
class Vocabulary:
    def __init__(self, min_freq=1):
        self.min_freq = min_freq
        
        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"
        self.sos_token = "<SOS>"
        self.eos_token = "<EOS>"
        
        self.special_tokens = [self.pad_token, self.unk_token, self.sos_token, self.eos_token]
        
        self.stoi = {token: idx for idx, token in enumerate(self.special_tokens)}
        self.itos = {idx: token for idx, token in enumerate(self.special_tokens)}
        
    def __len__(self):
        return len(self.stoi)
    
    def build_vocab(self, tokenized_sentences):
        counter = Counter(token for sentence in tokenized_sentences for token in sentence)
        
        idx = len(self.stoi)
        for token, freq in counter.items():
            if freq >= self.min_freq and token not in self.stoi:
                self.stoi[token] = idx
                self.itos[idx] = token
                idx += 1
                
    def numericalize(self, tokens):
        unk_idx = self.stoi[self.unk_token]
        return [self.stoi.get(token, unk_idx) for token in tokens]

In [ ]:
train_en_tokens = [pair[0] for pair in train_pairs]
train_es_tokens = [pair[1] for pair in train_pairs]

en_vocab = Vocabulary(min_freq=1)
es_vocab = Vocabulary(min_freq=1)

en_vocab.build_vocab(train_en_tokens)
es_vocab.build_vocab(train_es_tokens)

print(f"English Vocab Size: {len(en_vocab)}")
print(f"Spanish Vocab Size: {len(es_vocab)}")

In [ ]:
train_en_sentences = [pair[0] for pair in train_pairs]
train_es_sentences = [pair[1] for pair in train_pairs]

embedding_dim = 300
w2v_en_model = Word2Vec(sentences=train_en_sentences, vector_size=embedding_dim, window=5, min_count=1, workers=4)
w2v_es_model = Word2Vec(sentences=train_es_sentences, vector_size=embedding_dim, window=5, min_count=1, workers=4)

en_vectors = w2v_en_model.wv
es_vectors = w2v_es_model.wv

In [ ]:
vocab_size = len(en_vocab)
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in en_vocab.stoi.items():
    if word in en_vocab.special_tokens:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))
    elif word in en_vectors:
        embedding_matrix[idx] = en_vectors[word]
    else:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

embedding_tensor = torch.FloatTensor(embedding_matrix)

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, token_pairs, src_vocab, trg_vocab):
        self.token_pairs = token_pairs
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.token_pairs)

    def __getitem__(self, idx):
        src_tokens, trg_tokens = self.token_pairs[idx]
        
        src_indices = self.src_vocab.numericalize(src_tokens)
        
        trg_indices = (
            [self.trg_vocab.stoi[self.trg_vocab.sos_token]] + 
            self.trg_vocab.numericalize(trg_tokens) + 
            [self.trg_vocab.stoi[self.trg_vocab.eos_token]]
        )
        
        return torch.tensor(src_indices, dtype=torch.long), torch.tensor(trg_indices, dtype=torch.long)

In [ ]:
class CollateTranslation:
    def __init__(self, src_pad_idx, trg_pad_idx):
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx

    def __call__(self, batch):
        src_tensors = [item[0] for item in batch]
        trg_tensors = [item[1] for item in batch]
        
        src_padded = pad_sequence(src_tensors, batch_first=True, padding_value=self.src_pad_idx)
        trg_padded = pad_sequence(trg_tensors, batch_first=True, padding_value=self.trg_pad_idx)
        
        return src_padded, trg_padded

In [ ]:
class CustomGRUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # Combine gates (Reset gate "r", Update gate "z", New gate "n") into single layers for speed
        self.x_gates = nn.Linear(input_dim, 3 * hidden_dim)
        self.h_gates = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, x, h):
        # x shape: [batch_size, input_dim]
        # h shape: [batch_size, hidden_dim]
        
        # Project input and hidden states
        x_proj = self.x_gates(x)
        h_proj = self.h_gates(h)
        
        # Split into individual components
        x_r, x_z, x_n = torch.split(x_proj, self.hidden_dim, dim=1)
        h_r, h_z, h_n = torch.split(h_proj, self.hidden_dim, dim=1)
        
        # Calculate Reset and Update gates using standard sigmoid activation
        r = torch.sigmoid(x_r + h_r)
        z = torch.sigmoid(x_z + h_z)
        
        # Calculate the candidate hidden state using tanh activation
        n = torch.tanh(x_n + r * h_n)
        
        # Compute the final next hidden state
        h_next = (1 - z) * n + z * h
        
        return h_next

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        
        self.rnn_cell = CustomGRUCell(emb_dim, hidden_dim)
        
    def forward(self, src):
        batch_size = src.shape[0]
        src_len = src.shape[1]
        
        embedded = self.embedding(src)
        hidden = torch.zeros(batch_size, self.hidden_dim, device=src.device)
        
        for t in range(src_len):
            hidden = self.rnn_cell(embedded[:, t, :], hidden)
            
        return hidden.unsqueeze(0)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        
        self.rnn_cell = CustomGRUCell(emb_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, input_token, hidden):

        hidden = hidden.squeeze(0)
        
        embedded = self.embedding(input_token)
        
        hidden = self.rnn_cell(embedded, hidden)
        
        prediction = self.fc_out(hidden)
        
        return prediction, hidden.unsqueeze(0)

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.vocab_size
        
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        
        context_vector = self.encoder(src)
        hidden = context_vector
        
        decoder_input = trg[:, 0]
        
        for t in range(1, trg_len):
            prediction, hidden = self.decoder(decoder_input, hidden)
            
            outputs[:, t, :] = prediction
            
            top_prediction = prediction.argmax(1)
            
            is_teacher = random.random() < teacher_forcing_ratio
            decoder_input = trg[:, t] if is_teacher else top_prediction
            
        return outputs

In [ ]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    
    model.train()
    epoch_loss = 0
    
    progress_bar = tqdm(data_loader, desc="Training Batches", leave=False)
    
    for src, trg in progress_bar:
        src = src.to(device)
        trg = trg.to(device)
        
        optimizer.zero_grad()
        
        output = model(src, trg, teacher_forcing_ratio=0.5)
        
        output_dim = output.shape[-1]
        flat_output = output[:, 1:].reshape(-1, output_dim)
        flat_trg = trg[:, 1:].reshape(-1)
        
        loss = criterion(flat_output, flat_trg)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        progress_bar.set_postfix({"batch_loss": f"{loss.item():.4f}"})
        
    avg_loss = epoch_loss / len(data_loader)
    avg_perplexity = math.exp(avg_loss)
    
    return avg_loss, avg_perplexity

In [ ]:
def evaluate_epoch(model, data_loader, criterion, device):

    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for src, trg in data_loader:
            src = src.to(device)
            trg = trg.to(device)
            
            output = model(src, trg, teacher_forcing_ratio=0.0)
            
            output_dim = output.shape[-1]
            flat_output = output[:, 1:].reshape(-1, output_dim)
            flat_trg = trg[:, 1:].reshape(-1)
            
            loss = criterion(flat_output, flat_trg)
            
            epoch_loss += loss.item()
            
    avg_loss = epoch_loss / len(data_loader)
    avg_perplexity = math.exp(avg_loss)
    
    return avg_loss, avg_perplexity

In [ ]:
MAX_LEN = 40 
filtered_pairs = [
    (src, trg) for src, trg in train_pairs 
    if len(src) <= MAX_LEN and len(trg) <= MAX_LEN
]

# 2. Sort the pairs by the length of the source sentence
# This ensures sentences in the same batch are almost identical in length
filtered_pairs.sort(key=lambda x: len(x[0]))


train_dataset = TranslationDataset(train_pairs, en_vocab, es_vocab)
val_dataset = TranslationDataset(val_pairs, en_vocab, es_vocab)
test_dataset = TranslationDataset(test_pairs, en_vocab, es_vocab)

collate_fn = CollateTranslation(
    src_pad_idx=en_vocab.stoi[en_vocab.pad_token],
    trg_pad_idx=es_vocab.stoi[es_vocab.pad_token]
)


BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Created {len(train_loader)} training batches.")

In [ ]:
HIDDEN_DIM = 512
EMB_DIM = 300

encoder = Encoder(len(en_vocab), EMB_DIM, HIDDEN_DIM).to(device)
decoder = Decoder(len(es_vocab), EMB_DIM, HIDDEN_DIM).to(device)

model = Seq2Seq(encoder, decoder, device).to(device)

TRG_PAD_IDX = es_vocab.stoi[es_vocab.pad_token]

criterion = nn.CrossEntropyLoss(ignore_index=TRG_PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
N_EPOCHS = 10
best_valid_loss = float("inf")

print("Starting training pipeline...")
print("-" * 50)

for epoch in range(N_EPOCHS):
    train_loss, train_perp = train_epoch(model, train_loader, optimizer, criterion, device=device)
    
    valid_loss, valid_perp = evaluate_epoch(model, val_loader, criterion, device=device)
    
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "best_seq2seq_model.pt")
        
    print(f"Epoch: {epoch+1:02} | Train Loss: {train_loss:.4f} (Perplexity: {train_perp:.2f})")
    print(f"         | Val Loss:   {valid_loss:.4f} (Perplexity: {valid_perp:.2f})")
    print("-" * 50)